In [ ]:
import pandas as pd
import numpy as np

from IPython.display import display

from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
def read_file_DF(path_train, path_test):
    DF_train = pd.read_csv(path_train, sep=',')
    DF_test = pd.read_csv(path_test, sep=',')
    return DF_train, DF_test

def get_solutions(path_solutions):
    DF_solution = pd.read_csv(path_solutions, sep = ',')
    DF_solution = DF_solution['value']
    return DF_solution

def train_data_labels(DF_train):
    DF_train_y = DF_train['target']
    DF_train_x = DF_train.drop(columns='target')
    return DF_train_x, DF_train_y

def get_num_cat_col(DF_train_x):
    numeric_columns = DF_train_x.select_dtypes(include=['int64', 'float64']).columns
    categorical_columns = DF_train_x.select_dtypes(include = ['object']).columns
    return numeric_columns, categorical_columns

def drop_uselss_numeric_shit(DF_train_x, DF_test):
    cols_to_drop = ['cat_4', 'cat_5', 'cat_6']
    
    DF_train_x = DF_train_x.drop(columns=cols_to_drop, errors='ignore')
    DF_test = DF_test.drop(columns=cols_to_drop, errors='ignore')
    
    return DF_train_x, DF_test
    


def preprocess(categorical_columns, numeric_columns):
    num_preprocessing_pipeline = make_pipeline(
        KNNImputer(),
        PolynomialFeatures(),
        StandardScaler()
        )

    # pipeline for categorical data
    cat_preprocessing_pipeline = make_pipeline(
        SimpleImputer(strategy='most_frequent'),        # strategy = 'most_frequent'
        OneHotEncoder(handle_unknown='ignore')         # handle_unknown='ignore'
    )

    # connect the two pipelines
    preprocessing = ColumnTransformer(
        transformers = [
            ('categorical_data', cat_preprocessing_pipeline, categorical_columns),
            ('numeric_data', num_preprocessing_pipeline, numeric_columns)
        ]
    )
    return preprocessing

def reg_pipeline(preprocessing):
    regression_pipeline = make_pipeline(
        preprocessing,
        Ridge()
    )
    return regression_pipeline



def data_split_train_ev(DF_train_x, DF_train_y):
    x_train , x_eval, y_train, y_eval = train_test_split(DF_train_x, DF_train_y, test_size=0.2, random_state=42)
    return x_train , x_eval, y_train, y_eval



def grid(regression_pipeline):
        param_grid = {
                'columntransformer__numeric_data__knnimputer__n_neighbors' : [20, 30, 40],
                'columntransformer__numeric_data__polynomialfeatures__degree' : [2,3],

                'ridge__alpha' : [1100, 1300, 1500]
                
                # 'randomforestregressor__n_estimators': [200, 300, 400],
                # 'randomforestregressor__max_depth' : [5,10,15],
                # 'randomforestregressor__min_samples_leaf' : [3,5,10]
        }

        # perform the grid search
        grid_search = GridSearchCV(regression_pipeline, param_grid, cv = 4, scoring='r2', n_jobs=-1,) # return_train_score=True
        return grid_search

def grid_search_fit_pred(grid_search, x_train, y_train, x_eval):
        grid_search.fit(x_train, y_train)

        best_parameters = grid_search.best_params_
        best_score = grid_search.best_score_
        print(f"The best parameters are {best_parameters} with a score of {best_score}")

        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(x_eval)

        return y_pred, best_model



def evaluation(y_pred, y_eval):
    r2 = r2_score(y_pred, y_eval)
    MSE = mean_squared_error(y_pred, y_eval)
    return r2, MSE

def r2_training(x_train, y_train, best_model):
    # re-fit the model on x_train and y_train, then predict
    best_model.fit(x_train, y_train)
    pred = best_model.predict(x_train)
    r2_train = r2_score(pred, y_train)
    
    return r2_train



def plot(y_eval, y_pred):
    plt.scatter(y_eval, y_pred, alpha=0.3)
    plt.xlabel('True target')
    plt.ylabel('Predicted target')
    plt.title('Predicted vs True (eval set)')

    min_val = min(y_eval.min(), y_pred.min())
    max_val = max(y_eval.max(), y_pred.max())
        # plot a line that passes though min_val to max_val
        # min_val is the smallest value between y_eval and y_pred
        # max_val is the largest value between y_eval and y_pred

    plt.plot([min_val, max_val], [min_val, max_val])
    plt.show()
    pass



def pred_test_evaluate(best_model, DF_test, y_solution):
    # predict test data
    y_pred = best_model.predict(DF_test)
    
    # evaluate predictions
    # the professor gave only the first 100 predictions ... cunt
    n = len(DF_solution)
    y_pred_subset = y_pred[:n]

    r2_test = r2_score(y_solution, y_pred_subset)
    MSE_test = mean_squared_error(y_solution, y_pred_subset)
    MAE_test = mean_absolute_error(y_solution, y_pred_subset)

    return r2_test, MSE_test, MAE_test, y_pred_subset






if __name__ == '__main__':
    DF_train, DF_test= read_file_DF('../../../Dataset/LAB5/train_dataset.csv', '../../../Dataset/LAB5/test_dataset.csv')
    DF_solution = get_solutions('../../../Dataset/LAB5/solution.csv')
    DF_train_x, DF_train_y = train_data_labels(DF_train)
    DF_train_x, DF_test = drop_uselss_numeric_shit(DF_train_x, DF_test)
    numeric_columns, categorical_columns = get_num_cat_col(DF_train_x)
        
    preprocessing = preprocess(categorical_columns, numeric_columns)
    regression_pipeline = reg_pipeline(preprocessing)

    x_train , x_eval, y_train, y_eval = data_split_train_ev(DF_train_x, DF_train_y)

    grid_search = grid(regression_pipeline)
    y_pred, best_model = grid_search_fit_pred(grid_search, x_train, y_train, x_eval)

    r2_eval, MSE = evaluation(y_pred, y_eval)
    print(f"R2: {r2_eval} \nMSE: {MSE}")

    r2_train = r2_training(x_train, y_train, best_model)
    print(f"R2 score training: {r2_train}")
    print(f"R2 score evaluation: {r2_eval}")

    # predict test data
    r2_test, MSE_test, MAE_test, y_pred = pred_test_evaluate(best_model, DF_test, DF_solution)
    # print(f"Model's predictions: {y_pred}")
    #print(f"Actual target values: {DF_solution}")
    print(f"R2 score test: {r2_test}")

    plot(DF_solution, y_pred)

# STEPS

In [ ]:
train_data = pd.read_csv('train_dataset.csv')
test_data = pd.read_csv('test_dataset.csv')

In [ ]:
train_data.head()

In [ ]:
test_data.head()

In [ ]:
train_data.info()

# VERY IMPORTANT DATA SELECTION: ROWS
> I didn't do this but it would have been VERY helpful

- counts the Nan values per row
- sort the counted Nan values and shows the first 2 values
- prints number of rows with at least 15 missing values, 20 missing values and 25 missing values
- shows you how many rows you had before
- decides that rows with more than 20 missing values are dropped (a bit too strict to my mind)

In [ ]:
print("Top 20 rows with most missing values")

# counts Nan values on rows
row_na_counts = train_data.isna().sum(axis=1)
row_na_counts_sorted = row_na_counts.sort_values(ascending=False)
top_20_rows = row_na_counts_sorted.head(20)

print('row index \t Nan count')
print(top_20_rows)

# mind that each row can have up to 59 Nan values, one per column (59 columns)
rows_with_at_least_25_na = (row_na_counts >= 25).sum()
rows_with_at_least_20_na = (row_na_counts >= 20).sum()
rows_with_at_least_15_na = (row_na_counts >= 15).sum()

print(f"Number of rows with at least 25 missing values: {rows_with_at_least_25_na}")
print(f"Number of rows with at least 20 missing values: {rows_with_at_least_20_na}")
print(f"Number of rows with at least 15 missing values: {rows_with_at_least_15_na}")

##############################################################################################

total_rows_before = train_data.shape[0]
print(f"Total number of rows before removal: {total_rows_before}")

# mask_keep_rows = row_na_counts < 20
# train_data = train_data[mask_keep_rows].reset_index(drop=True)
train_data = train_data[train_data.isna().sum(axis=1) < 20 ].reset_index(drop=True)

total_rows_after = train_data.shape[0]
print(f"Total number of rows after removal: {total_rows_after}")

In [ ]:
continuous_features = []
for col in train_data.columns:
    if 'cont' in col:
        continuous_features.append(col)

# OR
# train_data.select_dtypes(include=['float64', 'int64']).columns

plt.figure(figsize=(20, 15))

for i, col in enumerate(continuous_features):
    subplot_index = i + 1
    plt.subplot(6, 5, subplot_index)            # builds a matrix 6,5 and specifies the index position (subplot_index) of the cell of the matrix to draw in
    sns.histplot(train_data[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()

This block builds a list of continuous feature names (those whose column name contains "cont") and then, for each of those columns, plots a histogram in a grid of subplots to inspect their distributions before scaling.

#### Useful only because you store the continuous/numeric features, the plot is just cool but kinda useless

In [ ]:
scaler = StandardScaler()

# Fit scaler on training continuous features and transform them
train_continuous = train_data[continuous_features]
train_continuous_scaled = scaler.fit_transform(train_continuous)
train_data[continuous_features] = train_continuous_scaled

# Transform test continuous features using the same scaler
test_continuous = test_data[continuous_features]
test_continuous_scaled = scaler.transform(test_continuous)
test_data[continuous_features] = test_continuous_scaled

# Fill remaining NaNs in continuous features with 0
train_data[continuous_features] = train_data[continuous_features].fillna(0)
test_data[continuous_features] = test_data[continuous_features].fillna(0)

This block standardizes the continuous features: it fits a StandardScaler on the training continuous columns, applies that scaling to both train and test, and then replaces any remaining NaNs in those continuous columns with 0.

I have no idea why the professor does this by itself, not in a pipeline and why it fills Nan with 0 anb not at least a simpleimputer??

In [ ]:
# Same plot a sbefore but with a KDE: Kernel Density Estimation --> looks cool, but useless

plt.figure(figsize=(20, 15))

for i, col in enumerate(continuous_features):
    subplot_index = i + 1
    plt.subplot(6, 6, subplot_index)
    sns.histplot(train_data[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()

This block again plots histograms (with KDE) for the continuous features, but now after scaling (and NaN filling). It lets you visually compare how the distributions changed after preprocessing.

In [ ]:
# Find ordinal features (column names containing 'ord')
ordinal_features = []
for col in train_data.columns:
    if 'ord' in col:
        ordinal_features.append(col)

# OR
# train_data.select_dtypes(include='object').columns

# Plot the distributions (counts) for each ordinal feature
plt.figure(figsize=(20, 15))

for i, col in enumerate(ordinal_features):
    subplot_index = i + 1
    plt.subplot(5, 4, subplot_index)
    
    # count categories
    value_counts = train_data[col].value_counts()

    # bar plot
    plt.bar(value_counts.index.astype(str), value_counts.values)
    plt.xticks(rotation=45)
    plt.title(col)

    plt.xticks(rotation=45)
    plt.title(col)

plt.tight_layout()
plt.show()

This block builds a list of ordinal feature names (all columns whose name contains 'ord'). Then it creates a 5×4 grid of subplots and, for each ordinal feature, plots a bar chart (countplot) of the category frequencies, rotating the x-axis labels for readability.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Configure an OrdinalEncoder:
# - unseen categories → -1
# - missing values    → -1
# - output type       → int

ordinal_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    encoded_missing_value=-1,
    dtype=int
)

# Fit on train ordinal columns and transform them
train_ordinal_data = train_data[ordinal_features]
train_ordinal_encoded = ordinal_encoder.fit_transform(train_ordinal_data)
train_data[ordinal_features] = train_ordinal_encoded

# Transform test ordinal columns with the same encoder
test_ordinal_data = test_data[ordinal_features]
test_ordinal_encoded = ordinal_encoder.transform(test_ordinal_data)
test_data[ordinal_features] = test_ordinal_encoded

print(f"Transformed {len(ordinal_features)} ordinal features using OrdinalEncoder")
print("Ordinal features:", ordinal_features)

This block creates an OrdinalEncoder that maps each category in the ordinal columns to an integer. Unknown categories and missing values are mapped to −1. It fits the encoder on the training ordinal columns, transforms them, and then applies the same transformation to the test ordinal columns. Finally, it prints how many ordinal features were transformed and their names.


#### OrdinalEncoder
> OrdinalEncoder is a scikit-learn transformer that converts categories into integers when the categories have a **meaningful order (ordinal)**.

Very simple idea:
- Input: columns with strings or labels like "low", "medium", "high".
- Output: numbers like 0, 1, 2 that keep the order.

Example:  
If it learns the order ["large", "medium", "small"] (just an example), it will map them to integers (e.g. 0, 1, 2). After encoding:  
- "large" → e.g. 0
- "medium" → e.g. 1
- "small" → e.g. 2

Key points:
- It outputs a numeric matrix usable by models.
- It treats categories as ordered (not one-hot).
- handle_unknown='use_encoded_value', unknown_value=-1 says: “If at test time you see a category not in training, encode it as −1.”
- encoded_missing_value=-1 means missing values are also mapped to −1.

In [ ]:
# Recompute ordinal_features list (optional but harmless)
ordinal_features = []
for col in train_data.columns:
    if 'ord' in col:
        ordinal_features.append(col)

plt.figure(figsize=(20, 15))

for i, col in enumerate(ordinal_features):
    subplot_index = i + 1
    plt.subplot(5, 4, subplot_index)
    sns.countplot(x=train_data[col])
    plt.title(col)

plt.tight_layout()
plt.show()

This block again collects the ordinal feature names and plots their distributions after encoding. Now the x-axis shows integer codes instead of original categories, so you can visually check the new encoded distributions.

In [ ]:
# Find categorical features (column names containing 'cat')
categorical_features = []
for col in train_data.columns:
    if 'cat' in col:
        categorical_features.append(col)

# Plot the distributions for each categorical feature
plt.figure(figsize=(20, 15))

for i, col in enumerate(categorical_features):
    subplot_index = i + 1
    plt.subplot(5, 4, subplot_index)
    sns.countplot(x=train_data[col])
    plt.xticks(rotation=45)
    plt.title(col)

plt.tight_layout()
plt.show()

# Inspect the list of categorical feature names
print("Categorical features:", categorical_features)

This block builds the list of categorical feature names (columns whose name contains 'cat'), then plots a 5×4 grid of bar charts showing category frequencies for each of those features. At the end it prints the list of categorical feature names.

In [ ]:
print(train_data.shape)                     # each column can have up to 6000 different values (every row with a different value)
train_data[categorical_features].nunique()
train_data.nunique()

In [ ]:
import numpy as np

print("cat_4 unique values:")
cat4_counts = train_data['cat_4'].value_counts(dropna=False)
print(cat4_counts)

print("\ncat_5 unique values:")
cat5_counts = train_data['cat_5'].value_counts(dropna=False)
print(cat5_counts)

print("\ncat_6 unique values:")
cat6_counts = train_data['cat_6'].value_counts(dropna=False)
print(cat6_counts)

# Drop these columns from train and test
cols_to_drop = ['cat_4', 'cat_5', 'cat_6']

train_data = train_data.drop(columns=cols_to_drop)
test_data = test_data.drop(columns=cols_to_drop)

# Update categorical_features list, removing the dropped columns
updated_categorical_features = []
for col in categorical_features:
    if col not in cols_to_drop:
        updated_categorical_features.append(col)

categorical_features = updated_categorical_features

# we applied OrdinalEncoder on ordinal features --> categories have a meaningful order
# we one-hot encode the categorical ones --> categories have no meaningful order
# One-hot encode remaining categorical features
train_data = pd.get_dummies(
    train_data,
    columns=categorical_features,
    drop_first=True
)

test_data = pd.get_dummies(
    test_data,
    columns=categorical_features,
    drop_first=True
)

This block prints the value counts of cat_4, cat_5, cat_6 to show that they’re essentially useless (often constant), drops them from both train and test, updates the list of categorical features to exclude them, and then applies one-hot encoding (get_dummies) to the remaining categorical columns in both train and test.


pd.get_dummies() is simply Pandas’ built-in One-Hot Encoder.  
Given a categorical column, e.g.:  
> color  
- red
- blue
- green
- blue

pd.get_dummies(df, columns=['color']) becomes:  

| color_blue | color_green | color_red |
|-----------|-------------|-----------|
| 0         | 0           | 1         |
| 1         | 0           | 0         |
| 0         | 1           | 0         |
| 1         | 0           | 0         |

Each category becomes a binary (0/1) column → that’s One-Hot Encoding.





In [ ]:
total_missing = train_data.isna().sum().sum()
print(total_missing)

This block computes the total number of missing values in the entire train_data DataFrame (summing over all columns and rows) and prints it. It’s a quick sanity check to see if any NaNs remain after preprocessing.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X = train_data.drop(columns=['target'])
y = train_data['target']

Splits the preprocessed train_data into:
- X: all columns except target (features)
- y: the target column (labels).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

models = {}

# Random Forest configuration
rf_model = RandomForestRegressor(random_state=42)
rf_params = {
    'n_estimators': [10, 20, 50],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5]
}

models['RandomForest'] = {
    'model': rf_model,
    'params': rf_params
}

# KNN configuration
knn_model = KNeighborsRegressor()
knn_params = {
    'n_neighbors': [5, 10],
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
models['KNN'] = {
    'model': knn_model,
    'params': knn_params
}

# Linear Regression configuration
lr_model = LinearRegression()
lr_params = {
    'fit_intercept': [True, False],
    'tol': [1e-4, 1e-3]
}
models['LinearRegression'] = {
    'model': lr_model,
    'params': lr_params
}

This block defines three different regression models (RandomForest, KNN, LinearRegression) and for each of them a dictionary of hyperparameters to explore with grid search. All are stored in a models dictionary so they can be iterated over later.

In [ ]:
from sklearn.model_selection import GridSearchCV

best_models = {}

for model_name, mp in models.items():
    print(f"Training {model_name}...")

    base_model = mp['model']
    param_grid = mp['params']

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=3,
        scoring='neg_mean_squared_error'
    )

    grid.fit(X, y)

    best_estimator = grid.best_estimator_
    best_params = grid.best_params_
    best_score = grid.best_score_      # negative MSE

    best_models[model_name] = best_estimator

    avg_mse = -best_score   # wtf man?
    print(f"{model_name} best params: {best_params}")
    print(f"{model_name} MSE: {avg_mse}\n")

This block runs a GridSearchCV for each model in models. For each one it:
- performs 3-fold CV using negative MSE as the score,
- finds the best hyperparameters,
- stores the best fitted model in best_models,
- and prints the best parameters and the corresponding mean MSE.

In [ ]:
chosen_model = best_models['KNN']

test_predictions = chosen_model.predict(test_data)

submission = pd.DataFrame()
submission['index'] = np.arange(len(test_predictions))
submission['value'] = test_predictions

submission.to_csv('submission.csv', index=False)

This block takes the best KNN model from the grid search, uses it to predict on the (preprocessed) test_data, and then creates a submission DataFrame with two columns: index and value. Finally, it writes the predictions to submission.csv in the required format.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

# Remove constant columns
# constant_cols = cols_to_drop
# train_data_clean = train_data.drop(columns=constant_cols)
# test_data_clean = test_data.drop(columns=constant_cols)

# Update categorical feature list after removing constant columns
# updated_categorical_features = []
# for col in categorical_features:
#     if col not in constant_cols:
#         updated_categorical_features.append(col)

# categorical_features = updated_categorical_features

# Build a ColumnTransformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        # numeric continuous features → scale
        ('num', StandardScaler(), continuous_features),

        # ordinal features → ordinal encode
        (
            'ord',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            ordinal_features
        ),

        # categorical features → one-hot encode (drop first level)
        (
            'cat',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore'
            ),
            categorical_features
        ),
    ],
    remainder='passthrough'   # leave any other columns as they are
)

This block prepares an “advanced” preprocessing pipeline: it removes constant columns, updates the list of categorical features accordingly, and creates a ColumnTransformer that:
- standardizes continuous numeric features,
- ordinal-encodes ordinal features,
- one-hot encodes remaining categorical features,

While passing through any leftover columns unchanged.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Compute absolute correlation of each continuous feature with the target (correlation can be compute only between numbers, so no ordinal and categorical features)
cont_corr = train_data[continuous_features].corrwith(
    train_data['target']
)
cont_corr_abs = cont_corr.abs()
cont_corr_sorted = cont_corr_abs.sort_values(ascending=False)

# Take the top 5 most correlated continuous features
top_continuous = cont_corr_sorted.head(5).index.tolist()

print(f"Top correlated continuous features: {top_continuous}")
print(f"Correlations: {cont_corr_sorted.head(5).values}")

This block computes the correlation between each continuous feature and the target, takes the absolute value, sorts features by |correlation| in descending order, and selects the top 5. It then prints which continuous features are most correlated with the target and their correlation magnitudes, **presumably to later build polynomial features only on these most informative numeric variables**.

#### PolynomialFeatures explodes the number of features.
- If you take 30 continuous features and do degree-3 polynomials, you suddenly get hundreds / thousands of columns (all squares, cubes, interactions…).
- That’s heavy, slow, and increases overfitting risk.

So instead of doing polynomials on all continuous features, you can do them only on the most promising ones.  
- “Most promising” here = features that have the highest absolute correlation with the target.
- If cont_7, cont_13, cont_2, cont_26, cont_29 are the top 5 by |corr|, you treat them as “the stars”.